# Custom GluonTS Model

This notebook adapts the model outlined in [TEMPO](chrome-extension://efaidnbmnnnibpcajpcglclefindmkaj/https://arxiv.org/pdf/2310.04948) to [GluonTS](https://ts.gluon.ai/stable/index.html) and follows GluonTS's [Custom Model Tutorial](https://ts.gluon.ai/stable/tutorials/advanced_topics/howto_pytorch_lightning.html). There are two main tasks we need to complete to adapt TEMPO to GluonTS:
1. ~~Create a [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) wrapper around TEMPO so we can perform training~~
2. ~~Create a GluonTS [`Predictor`](https://ts.gluon.ai/stable/api/gluonts/gluonts.torch.model.predictor.html?highlight=predictor#module-gluonts.torch.model.predictor) so we can perform inference~~

## Dataset

Load the electricity dataset

In [16]:
from gluonts.dataset.repository import get_dataset
from datetime import datetime

format = "%m/%d/%Y %I:%M:%S%p"


def print_timestamp():
    now = datetime.now()
    formatted_time = now.strftime(format)
    print(f"Last time ran: {formatted_time}")


dataset = get_dataset("electricity")
print_timestamp()

Last time ran: 03/04/2025 02:36:04PM


## Training

### PyTorch Lightning Wrapper

Before creating the PyTorch wrapper, we'll define the hyperparameters we need for training.

In [17]:
learning_rate = 1e-3
batch_size = 128
num_batches_per_epoch = 50
max_epochs = 1
prediction_length = 96

# Number of past samples to use when computing forecasts
context_length = 2 * 7 * 24

print_timestamp()

Last time ran: 03/04/2025 02:36:04PM


Create a [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/) wrapper to wrap around TEMPO so we can train the model using PyTorch Lightning.

In [18]:
import pytorch_lightning as pl
import torch
from gluonts.torch.distributions import StudentTOutput

from tempo.models.TEMPO import TEMPO


class LightningTEMPO(TEMPO, pl.LightningModule):
    def __init__(self, configs, args=None):
        super().__init__(configs)
        # TODO: change args default value so it's not None once you finish your prototype
        # Commmand line arguments
        self.args = args

        # Model configuration
        self.configs = configs

        # TODO: once you get a prototype working, change the code to allow for different output distributions
        # Type of distribution for model's output. We'll use a Student's t-distribution
        self.distr_output = StudentTOutput()

    def training_step(self, batch, batch_index):
        """
        Defines the logic for a single training loop iteration.
        """
        # Get past time series values
        past_target = batch["past_target"]

        # Get future time series values
        future_target = batch["future_target"]

        # TODO: figure out how to get trend, seasonal, and residual components from custom GluonTS datasets
        # Compute forward pass to get Student's t-distribution arguments
        (
            distr_args,  # parameters for students t distribution
            loc,
            scale,
        ) = self(past_target=past_target)

        # Create Student's t-distribution
        student_t_distr = self.distr_output.distribution(distr_args)

        # TODO: once you get a prototype working, change the code to compute different losses based on output distribution
        # Compute Student's t negative log-likelihood loss
        loss = -student_t_distr.log_prob(future_target)

        return loss.mean()

    # ? Not sure if we need this method
    def validation_step(self, batch, batch_index):
        """
        Defines the logic for a single validation loop iteration.
        """
        pass

    # ? Not sure if we need this method
    def test_step(self, batch, batch_index):
        """
        Defines the logic for a single test loop iteration.
        """
        pass

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        return optimizer


print_timestamp()

Last time ran: 03/04/2025 02:36:04PM


### Training Dataloader

Before creating the training set's dataloader, we'll do some preprocessing.

In [19]:
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddObservedValuesIndicator,
    InstanceSplitter,
    ExpectedNumInstanceSampler,
)

# Impute `nan`s in the target field with 0 and add a field indicating which
# values were imputed.
mask_unobserved = AddObservedValuesIndicator(
    target_field=FieldName.TARGET,
    output_field=FieldName.OBSERVED_VALUES,
)

# Split instances in the trianing set.
instance_sampler = ExpectedNumInstanceSampler(
    num_instances=1,
    min_future=prediction_length,
)

training_splitter = InstanceSplitter(
    target_field=FieldName.TARGET,
    is_pad_field=FieldName.IS_PAD,
    start_field=FieldName.START,
    forecast_start_field=FieldName.FORECAST_START,
    instance_sampler=instance_sampler,
    past_length=context_length,
    future_length=prediction_length,
    time_series_fields=[FieldName.OBSERVED_VALUES],
)

print_timestamp()

Last time ran: 03/04/2025 02:36:04PM


Once we're done with preprocesing, we'll create the training set's dataloader.

In [20]:
from gluonts.dataset.loader import TrainDataLoader
from gluonts.torch.batchify import batchify

data_loader = TrainDataLoader(
    dataset.train,
    batch_size=batch_size,
    stack_fn=batchify,
    transform=mask_unobserved + training_splitter,
    num_batches_per_epoch=num_batches_per_epoch,
)

print_timestamp()

Last time ran: 03/04/2025 02:36:04PM


Now that we have a PyTorch Lightning wrapper and a training dataloader, we can instantiate the model and train it.

In [ ]:
from omegaconf import OmegaConf
from pytorch_lightning import Trainer

print_timestamp()

# Load model configuration
configs = OmegaConf.load("./configs/run_TEMPO.yml")

# Initialize model wrapped with PyTorch Lightning
model = LightningTEMPO(configs)

# Initialize PyTorch Lightning trainer
trainer = Trainer(max_epochs=max_epochs)

# Train model
trainer.fit(model, data_loader)

------------------No need to load pretrained GPT model------------------


/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/peft/tuners/lora/layer.py:1150: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/mike_gee/miniconda3/envs/tempo/lib/python3.8/s ...
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/si

Trainable params: 308736 || All params: 82207488
Trainable params: 308736 || All params: 82207488
Trainable params: 308736 || All params: 82207488


Training: |          | 0/? [00:00<?, ?it/s]

## Inference

### Creating a GluonTS Predictor

First, we'll split the instances in the test set.

In [ ]:
from gluonts.transform import TestSplitSampler

prediction_splitter = InstanceSplitter(
    target_field=FieldName.TARGET,
    is_pad_field=FieldName.IS_PAD,
    start_field=FieldName.START,
    forecast_start_field=FieldName.FORECAST_START,
    instance_sampler=TestSplitSampler(),
    past_length=context_length,
    future_length=prediction_length,
    time_series_fields=[FieldName.OBSERVED_VALUES],
)

print_timestamp()

Then, we'll get a [`Predictor`](https://ts.gluon.ai/stable/api/gluonts/gluonts.torch.model.predictor.html?highlight=predictor#module-gluonts.torch.model.predictor) from our model using the `get_predictor()` method and use it to compute forecasts.

In [ ]:
predictor = model.get_predictor(mask_unobserved + prediction_splitter)

print_timestamp()

### Model Evaluation

Now that we have a `Predictor`, we can compute predictions on the test set and evaluate our model. We can use GluonTS's [`make_evaluation_predictions()`](https://ts.gluon.ai/stable/api/gluonts/gluonts.evaluation.backtest.html?highlight=make_evaluation_predictions#gluonts.evaluation.backtest.make_evaluation_predictions) function to automate the process of computing test set predictions and model evaluation.

In [ ]:
from gluonts.evaluation import make_evaluation_predictions

"""
make_evaluation_predictions() returns a pair of iterators
- the first iterator contains the predicted values
- the second iterator contains the ground truth values
"""
forecast_iterator, ground_truth_iterator = make_evaluation_predictions(
    dataset=dataset.train,
    predictor=predictor,
)

# List of forecasts
forecasts_list = list(forecast_iterator)

# List of ground truth time series values
ground_truth_list = list(ground_truth_iterator)

print_timestamp()

Create a plot of the forecast and ground truth values for a single time series.

In [ ]:
import matplotlib.pyplot as plt

# Get the first ground truth time series in the test set
ground_truth = ground_truth_list[0]

# Get the corresponding forecast
forecast = forecasts_list[0]

# Create plot
plt.plot(ground_truth[-150:].to_timestamp())
forecast.plot(show_label=True)
plt.legend()

print_timestamp()

We'll use the [`Evaluator`](https://ts.gluon.ai/stable/api/gluonts/gluonts.evaluation.html?highlight=evaluator#gluonts.evaluation.Evaluator) class to evaluate our model.

In [ ]:
from gluonts.evaluation import Evaluator

print_timestamp()

evaluator = Evaluator(quantiles=[0.1, 0.5, 0.9])
aggregated_metrics, item_metrics = evaluator(ground_truth_list, forecasts_list)

Running evaluation: 321it [00:00, 854.30it/s]
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/pandas/core/dtypes/astype.py:138: UserWarning: Warning: converting a masked element to nan.
  return arr.astype(dtype, copy=True)
/home/mike_gee/miniconda3/envs/tempo/lib/python3.8/site-packages/pandas/core/dtypes/astype.py:138: UserWarning: Warning: converting a masked element to nan.
  return arr.astype(dtype, copy=True)


- `aggregated_metrics` is a dictionary containing metrics aggregated across time steps and time series
  - These metrics evaluate the overall forecast quality across all time series

In [ ]:
import json

print_timestamp()
print(json.dumps(aggregated_metrics, indent=4))

{
    "MSE": 135144542.95538032,
    "abs_error": 68301208.0,
    "abs_target_sum": 68301208.0,
    "abs_target_mean": 2216.4203011422637,
    "seasonal_error": 189.82371656867073,
    "MASE": 11.53922772651395,
    "MAPE": 1.0,
    "sMAPE": 2.0,
    "MSIS": 461.56910911049914,
    "num_masked_target_values": 0.0,
    "QuantileLoss[0.1]": 13660241.6,
    "Coverage[0.1]": 0.010449117341640708,
    "QuantileLoss[0.5]": 68301208.0,
    "Coverage[0.5]": 0.010449117341640708,
    "QuantileLoss[0.9]": 122942174.4,
    "Coverage[0.9]": 0.010449117341640708,
    "RMSE": 11625.168512988546,
    "NRMSE": 5.245019866943716,
    "ND": 1.0,
    "wQuantileLoss[0.1]": 0.19999999999999998,
    "wQuantileLoss[0.5]": 1.0,
    "wQuantileLoss[0.9]": 1.8,
    "mean_absolute_QuantileLoss": 68301208.0,
    "mean_wQuantileLoss": 1.0,
    "MAE_Coverage": 0.4895508826583593,
    "OWA": NaN
}


- `item_metrics` is a pandas DataFrame containing per-series metrics
  - Each row corresponds to a single time series

In [ ]:
print_timestamp()
item_metrics.head()

Last time ran: 03/04/2025 02:36:00PM


,item_id,forecast_start,MSE,abs_error,abs_target_sum,abs_target_mean,seasonal_error,MASE,MAPE,sMAPE,num_masked_target_values,ND,MSIS,QuantileLoss[0.1],Coverage[0.1],QuantileLoss[0.5],Coverage[0.5],QuantileLoss[0.9],Coverage[0.9]
0,0,2014-05-22 20:00,202.802083,851.0,851.0,8.864583,8.025951,1.104490,1.0,2.0,0.0,1.0,44.179603,170.2,0.000000,851.0,0.000000,1531.8,0.000000
1,1,2014-05-22 20:00,11042.822917,9943.0,9943.0,103.572917,9.589706,10.800427,1.0,2.0,0.0,1.0,432.017086,1988.6,0.000000,9943.0,0.000000,17897.4,0.000000
2,2,2014-05-22 20:00,64.000000,752.0,752.0,7.833333,8.707561,0.899601,1.0,2.0,0.0,1.0,35.984054,150.4,0.020833,752.0,0.020833,1353.6,0.020833
3,3,2014-05-22 20:00,193520.416667,40990.0,40990.0,426.979167,48.403795,8.821192,1.0,2.0,0.0,1.0,352.847680,8198.0,0.000000,40990.0,0.000000,73782.0,0.000000
4,4,2014-05-22 20:00,29245.000000,15984.0,15984.0,166.500000,25.397152,6.555853,1.0,2.0,0.0,1.0,262.234132,3196.8,0.000000,15984.0,0.000000,28771.2,0.000000
